# TIL — 2026-11-12

> Chapter 09 「React Mini Pop-up Store」 STEP 06~19: DB 모델·Seed → FastAPI `GET /products`(Repository/Service/Router) → pytest → Next.js 상품 카드 목록

## 오늘 한 일

| STEP | 내용 | 상태 |
| --- | --- | --- |
| 06 | `orders` / `order_items` / `products` 관계 설계 | 완료 |
| 07 | SQLAlchemy 모델 3개 추가 + 실제 DB에 테이블 생성 | 완료 (DB에서 확인) |
| 08 | Seed 상품 6개 입력 | 완료 (SELECT로 확인) |
| 09 | Swagger 접속 | 완료 (`/docs` 200) |
| 10 | Backend와 DB 클라이언트가 같은 데이터를 읽는지 대조 | 완료 |
| 11 | Product Repository | 완료 |
| 12 | Product Service | 완료 |
| 13 | `GET /products` | 완료 (HTTP 200 / 배열 / 필드) |
| 14 | pytest로 `GET /products` 검증 | 완료 (5 passed) |
| 15 | Next.js 개발 서버 실행 | 서버 응답 확인, 브라우저 Console은 직접 확인 필요 |
| 16 | `ProductCard` 구조 | 완료 (tsc/eslint 통과) |
| 17 | 카드 1개 표시 | 화면 확인 |
| 18 | 같은 컴포넌트로 서로 다른 상품 2개 | 서버 HTML로 확인 |
| 19 | `map()` + `key`로 6개 렌더 | 서버 HTML로 확인, **Playwright Smoke ①은 HOLD** |

## 1. 테이블 설계 (STEP 06)

```
orders(1) ──< order_items(N) >── products(1)
```

- 쿠키 2개 + 음료 1개를 한 번에 주문하면 `orders` 1행 + `order_items` 2행. 상품 종류마다 상세 한 행.
- `order_items.price`는 **주문 당시 단가 스냅샷**. 나중에 `products.price`가 바뀌어도 이미 한 주문 금액은 유지되어야 함.
- 장바구니는 React State로만 보관하고 별도 테이블은 만들지 않음.

## 2. SQLAlchemy 모델과 테이블 생성 (STEP 07)

- 기존 `backend/models.py`에 `Product`, `Order`, `OrderItem`을 추가. 기존 `Member`와 `Base`는 그대로.
- 제약조건: `price >= 0`, `stock >= 0`, `quantity >= 1`, FK 2개(`order_id`, `product_id`), 숫자 컬럼 NOT NULL.
- `orders.status` 기본값 `'created'`, `orders.created_at` 기본값 `now()` (`server_default`).
- 새 테이블은 글의 확인 SQL에 맞춰 `public` 스키마. (기존 `Member`는 `practice` 스키마라 스키마를 모델마다 명시함)
- 테이블 생성은 **새 세 테이블만** `Base.metadata.create_all(engine, tables=[...])`. 삭제·초기화 없음. 같은 이름 테이블이 이미 있으면 중단하도록 먼저 확인.
- 실제 DB에서 `information_schema`와 `pg_constraint`로 컬럼·PK/FK/CHECK를 조회해 확인.

## 3. Seed (STEP 08)

- `backend/seed_products.py`: 이미 있는 **이름은 건너뛰고** 없는 것만 insert → 재실행해도 중복 없음. 삭제 없음.
- 실행은 `practice/phase03`에서 `python -m backend.seed_products` (`-I` 옵션을 쓰면 `backend` 모듈을 못 찾음).
- 결과: 1회차 추가 6 / 건너뜀 0, 2회차 추가 0 / 건너뜀 6. `SELECT`로 6행 확인.

## 4. Router → Service → Repository → DB (STEP 10~13)

- **STEP 10**: SQLAlchemy `SessionLocal`로 읽은 결과와 psycopg로 따로 연결해서 읽은 결과를 대조 → DB `postgres` / 스키마 `public` / 건수 6 / 6행 모두 일치.
- **STEP 11** `repositories/product_repository.py`: `find_all(db)` — `select(Product).order_by(Product.id)`. 기존 `member_repository.py` 스타일(함수형, `db: Session` 인자)에 맞춤. 정렬을 명시하지 않으면 순서가 보장되지 않음.
- **STEP 12** `services/product_service.py`: `get_products_service(db)` → `product_repository.find_all(db)`. 글의 `ProductService.list_products()`는 의사 코드라 프로젝트 관례(`get_members_service`)에 맞춰 이름을 정함. Service 안에서 Engine/Session을 새로 만들지 않음.
- **STEP 13**: `schemas.py`에 `ProductResponse`(`from_attributes=True`), `routers/products.py`, `main.py`에 `include_router`.
- 결과: `GET /products` → 200, 배열 길이 6, 필드는 `id/name/price/category/stock`뿐, price·stock은 정수, CORS 헤더도 붙음.

## 5. pytest와 테스트 격리 (STEP 14)

- 실제 학습용 DB의 상품이 6개라는 사실을 테스트에 고정하면 환경마다 깨짐 → **인메모리 SQLite**로 테스트 전용 DB를 만듦.
- 모델의 스키마(`public`/`practice`)는 SQLite에 없으므로 `execution_options(schema_translate_map={"public": None, "practice": None})`로 변환.
- `backend.db`는 import 시점에 `DB_*` 환경변수를 요구 → `conftest.py`에서 더미 값을 먼저 `os.environ`에 넣어 실제 `.env`·실제 DB를 쓰지 않게 함.
- `app.dependency_overrides[get_db]`로 테스트 Session을 주입. "테스트 DB에만 있는 상품이 응답에 나온다"로 override를 검증.
- 테스트 5개: 200 / JSON 배열 / 필드·타입 계약 / 빈 목록 200 + `[]` / 테스트 Session 사용 → `5 passed`.
- 테스트 후에도 실제 DB `products`는 6개 그대로.

## 6. 이번에 만난 환경 문제 (Windows 앱 제어 정책)

- 컴파일된 `.dll`/`.pyd`가 막힘 → **psycopg 3.3.6 고정**, **SQLAlchemy 2.0.x**(2.1은 컴파일 모듈이 막힘).
- `pip install -U`로 올리면 다시 막힐 수 있으니 주의. (어제 TIL에도 있는 내용)
- 터미널 한글이 깨져 보여도 저장된 값은 정상일 수 있음 → `ascii(...)`로 찍어서 유니코드 이스케이프로 확인. (curl 응답을 `python`의 stdin으로 읽다가 CP949로 잘못 디코딩해 또 깨져 보였음 → `sys.stdin.buffer.read().decode("utf-8")`)
- Git Bash에서 `curl ... -w "$p -> ..."`의 `/`로 시작하는 문자열이 `C:/Program Files/Git/`로 바뀌어 출력됨(경로 변환). 요청 자체는 정상.

## 7. 프론트: 상품 카드 (STEP 15~19)

- 개발 서버: `member-next-app`에서 `npm run dev` → `http://localhost:3000`. 시작하면서 Next 16이 `AGENTS.md`를 자동 생성함(`agentRules: false`로 끌 수 있음).
- `types/product.ts`: STEP 13 API 계약과 같은 `Product` 타입.
- `components/ProductCard.tsx`: `product`와 `onAdd` Props, 가격은 `toLocaleString("ko-KR") + "원"`, `stock <= 0`이면 "품절" + 버튼 `disabled`. 카드는 장바구니 배열을 갖지 않고 이벤트만 부모로 전달.
- **`"use client"`가 필요한 이유**: `onClick`(이벤트 핸들러)과 함수 props는 서버 컴포넌트에서 쓸 수 없음. 서버 컴포넌트 전체를 Client로 바꾸는 게 아니라 필요한 컴포넌트에만 선언.
- `app/store/page.tsx`: 처음엔 카드 1개 → 2개 → 배열 6개를 `map()`으로 표시. `key={product.id}` (index 대신 id). 임시 배열이라 STEP 25에서 실제 API로 교체 예정.
- SSR HTML에서 `3,000<!-- -->원`으로 보임: React가 연속된 텍스트 사이에 넣는 표시용 주석이고 화면에서는 "3,000원".
- 다크 모드에서 제목 `#0f172a`가 배경에 묻힘 → 제목의 고정 색을 빼고 주변 글자색을 따르게 함. `/members` 화면에도 같은 문제가 있을 수 있음.
- 처음 404가 났던 건 접속 주소나 개발 서버를 띄운 폴더가 달랐던 경우가 많음(서버 터미널의 폴더·포트 확인).

## 검증 상태 (솔직하게)

- ✅ DB: 세 테이블·제약조건·Seed 6행을 실제 조회로 확인
- ✅ Backend: Repository/Service/`GET /products` 실제 호출, STEP 10 대조 일치
- ✅ pytest 5 passed (실제 DB 불변 확인)
- ✅ 프론트 `tsc --noEmit`, `eslint` 통과, `/store` 서버 HTML에 카드 6개·버튼 6개
- ⬜ 브라우저 F12 Console의 오류·key 경고 확인 (직접 확인 필요)
- ⬜ **Playwright Smoke ① 미설치 → HOLD** (설치 시 패키지 + 브라우저 다운로드가 필요하고, 이 PC는 실행 파일 차단 가능성이 있음)
- 강의 글 기준으로, 실제 실행해서 확인하기 전에는 PASS로 적지 않기.

## 헷갈렸던 점 / 배운 점

- 글의 의사 코드(`ProductService.list_products()`)를 그대로 쓰기보다 **기존 프로젝트의 관례(함수형, `db: Session` 인자)에 맞추는 게** 맞았음.
- 테스트 격리는 "실제 DB의 개수를 고정"하는 게 아니라 **알려진 가상 데이터를 따로 준비**하는 것.
- "서버가 떴다"와 "화면이 정상이다"는 다름. 서버 HTML로는 렌더링 결과까지만 확인되고, Console 오류는 브라우저에서 봐야 함.

## TODO

- [ ] 브라우저에서 `/store`의 Console 오류·key 경고 확인
- [ ] Playwright를 설치할지 결정(Smoke ①) — 아니면 HOLD로 유지
- [ ] `/members` 화면의 다크 모드 제목 색 정리 여부 결정
- [ ] `member-next-app`(현재 `assets/images/phase03` 아래)과 `practice/phase03` 위치 정리
- [ ] 루트 `.gitignore`에 `.venv/`, `node_modules/` 추가
- [ ] `assets/images/phase03/.env` 원본 정리(비밀번호 파일이 추적되는 폴더에 있음)
- [ ] STEP 20 이후 진행 (STEP 25에서 임시 배열을 `GET /products` 응답으로 교체)
